# Panel 1A: Auditoría de Calidad y Limpieza Profunda de Datos (Data Quality & Sanitization)

Este cuaderno constituye la **Fase 1 del Pipeline Analítico** del proyecto **SmartBazar**. Su propósito fundamental es prevenir el antipatrón *Garbage In, Garbage Out* sometiendo los datos crudos (`datasets/crudo/`) a una exhaustiva auditoría y saneamiento antes de utilizarlos en análisis exploratorios o modelos de Machine Learning.

### Objetivos de la Auditoría y Limpieza:
1. **Inspección de Datos Crudos:** Visualizar las primeras filas (`head()`), estructura (`info()`), conteo de valores nulos (`isnull().sum()`) y medias/estadísticos (`describe()`).
2. **Levantamiento de Observaciones Críticas de Negocio:**
   - **Fechas y Horario (Registro por Lote):** Normalizar los distintos formatos de fecha y **excluir la variable hora** al identificarse que las ventas se digitan los fines de semana (por lo que la hora refleja la digitación y no la transacción real).
   - **Detalle de Ventas:** Eliminar columnas vacías generadas por exportación (`Unnamed`) y sanear descripciones.
   - **Inventario:** Eliminar filas vacías, corregir **stocks negativos** provocados por ventas sin ingreso registrado en kardex e imputar el **Stock Mínimo** diferenciado por departamento (evitando el valor por defecto genérico de 5).
3. **Demostración Before vs. After y Exportación:** Verificación final de calidad y exportación del insumo oficial a `datasets/limpio/`.

In [ ]:
import pandas as pd
import numpy as np
import os

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 1000)
print("Entorno configurado para auditoría de calidad de datos.")

## 1. Auditoría de Entrada: Inspección de Datos Crudos (`datasets/crudo/`)
Cargamos los archivos tal como salieron del sistema fuente para diagnosticar su calidad inicial (*Garbage In*).

In [ ]:
crudo_dir = 'datasets/crudo'

df_ventas_raw = pd.read_csv(os.path.join(crudo_dir, 'ventas.csv'), sep=';', encoding='utf-8-sig')
df_detalle_raw = pd.read_csv(os.path.join(crudo_dir, 'detalle_ventas.csv'), sep=';', encoding='utf-8-sig', skiprows=1)
df_inv_raw = pd.read_csv(os.path.join(crudo_dir, 'inventario.csv'), sep=';', encoding='utf-8-sig', skiprows=1)

print("=== AUDITORÍA INICIAL: VENTAS CRUDAS ===")
print("Dimensiones:", df_ventas_raw.shape)
print("\nPrimeras 5 filas crudas (df.head()):")
print(df_ventas_raw.head())
print("\nValores nulos en ventas crudas:")
print(df_ventas_raw.isnull().sum())

In [ ]:
print("=== AUDITORÍA INICIAL: DETALLE DE VENTAS Y COLUMNAS FANTASMA ===")
print("Dimensiones crudas:", df_detalle_raw.shape)
print("Columnas presentes:", df_detalle_raw.columns.tolist())
print("\nConteo de nulos por columna:")
print(df_detalle_raw.isnull().sum())
print("\nPrimeras 3 filas del detalle crudo:")
print(df_detalle_raw.head(3))

In [ ]:
print("=== AUDITORÍA INICIAL: INVENTARIO Y FILAS VACÍAS ===")
print("Dimensiones crudas:", df_inv_raw.shape)
print("\nConteo de nulos por columna:")
print(df_inv_raw.isnull().sum())
print("\nEstadísticas descriptivas (media, min, max) de variables numéricas crudas:")
print(df_inv_raw[['Stock_Minimo', 'Stock_Actual', 'Costo_Unitario', 'Precio_Venta']].describe(include='all'))

## 2. Levantamiento de Observaciones y Saneamiento Específico

### Observación 1: Fechas con Formatos Mixtos y Hora no Representativa
- **Diagnóstico:** En `ventas.csv` coexisten fechas con formato corto (`DD/MM/YY`) y formato largo (`M/D/YYYY HH:MM:SS`). Más importante aún, **la hora no representa el momento de compra real**, ya que el bazar digita los comprobantes por lotes los fines de semana.
- **Acción de Limpieza:** Normalizar la columna `Fecha` al formato canónico `YYYY-MM-DD` y **descartar el uso de la hora** para clustering o análisis de demanda horaria.

In [ ]:
print("Muestras de Fecha antes de normalizar:", df_ventas_raw['Fecha'].unique()[:5])

# Limpieza de Ventas
df_ventas_clean = df_ventas_raw.dropna(subset=['ID', 'Total']).copy()
df_ventas_clean['Fecha'] = pd.to_datetime(df_ventas_clean['Fecha'], format='mixed', errors='coerce').dt.strftime('%Y-%m-%d')
df_ventas_clean['Metodo_Pago'] = df_ventas_clean['Metodo_Pago'].astype(str).str.strip().str.upper()
df_ventas_clean['Total'] = pd.to_numeric(df_ventas_clean['Total'], errors='coerce').fillna(0.0)

print("\nMuestras de Fecha normalizada (YYYY-MM-DD sin ruido horario):")
print(df_ventas_clean[['ID', 'Fecha', 'Metodo_Pago', 'Total']].head())

### Observación 2: Saneamiento de Detalle de Ventas
- **Diagnóstico:** Existen 4 columnas vacías (`Unnamed: 0`, `Unnamed: 10`, `Unnamed: 11`, `Unnamed: 12`) y descripciones con espacios en blanco o faltantes.
- **Acción de Limpieza:** Filtrar columnas `Unnamed`, sanear `ID_Venta` y convertir numéricos preservando solo transacciones válidas.

In [ ]:
df_detalle_clean = df_detalle_raw.loc[:, ~df_detalle_raw.columns.str.contains('^Unnamed')].copy()
df_detalle_clean = df_detalle_clean.dropna(subset=['ID_Venta']).copy()
df_detalle_clean['ID_Venta'] = df_detalle_clean['ID_Venta'].astype(str).str.strip()
df_detalle_clean['Fecha'] = pd.to_datetime(df_detalle_clean['Fecha'], format='mixed', errors='coerce').dt.strftime('%Y-%m-%d')
df_detalle_clean['Descripcion'] = df_detalle_clean['Descripcion'].fillna('SIN DESCRIPCION').astype(str).str.strip()

for col in ['Cantidad', 'Precio_Unitario', 'Subtotal']:
    df_detalle_clean[col] = pd.to_numeric(df_detalle_clean[col].astype(str).str.replace(',', '').str.strip(), errors='coerce').fillna(0.0)
    df_detalle_clean[col] = df_detalle_clean[col].clip(lower=0)

print("Dimensiones detalle saneado:", df_detalle_clean.shape)
print("Nulos restantes en detalle:", df_detalle_clean.isnull().sum().sum())

### Observación 3: Inventario - Filas Vacías, Stocks Negativos y Stock Mínimo Diferenciado
- **Diagnóstico 3A:** 511 filas en `inventario.csv` son completamente nulas/vacías.
- **Diagnóstico 3B (Stocks Negativos):** Artículos de alta rotación (ej. `HOJA DE COLORES` con -184) presentan stock negativo por ventas realizadas antes del registro de entrada en almacén.
- **Diagnóstico 3C (Stock Mínimo):** El 89% tiene `Stock_Minimo` en nulo y lo existente está sesgado al valor 5.
- **Acción de Limpieza:**
  1. Eliminar las 511 filas vacías.
  2. Ajustar stocks negativos a `0` físico creando la bandera de auditoría `Alerta_Kardex_Negativo`.
  3. Imputar `Stock_Minimo` por departamento: **5** para Útiles/Librería y **2** para Fotocopiadora/Servicios.

In [ ]:
df_inv_clean = df_inv_raw.loc[:, ~df_inv_raw.columns.str.contains('^Unnamed')].copy()
df_inv_clean = df_inv_clean.dropna(subset=['ID']).copy()

for col in ['Costo_Unitario', 'Precio_Venta']:
    df_inv_clean[col] = pd.to_numeric(df_inv_clean[col].astype(str).str.replace(',', '').str.strip(), errors='coerce').fillna(0.0)
for col in ['Stock_Minimo', 'Stock_Actual']:
    df_inv_clean[col] = pd.to_numeric(df_inv_clean[col], errors='coerce')

# Identificar stocks negativos
negativos = df_inv_clean[df_inv_clean['Stock_Actual'] < 0]
print(f"Artículos detectados con Stock_Actual negativo (desfase de kardex): {len(negativos)}")
print(negativos[['ID', 'Descripcion', 'Departamento', 'Stock_Actual']].head(5))

# Saneamiento
df_inv_clean['Stock_Minimo'] = df_inv_clean.apply(
    lambda r: 5 if pd.isna(r['Stock_Minimo']) and str(r['Departamento']).upper() == 'UTILES'
    else (2 if pd.isna(r['Stock_Minimo']) else r['Stock_Minimo']),
    axis=1
).astype(int)

df_inv_clean['Alerta_Kardex_Negativo'] = df_inv_clean['Stock_Actual'] < 0
df_inv_clean['Stock_Actual'] = df_inv_clean['Stock_Actual'].fillna(0).clip(lower=0).astype(int)

print("\nDistribución final de Stock_Minimo saneado por Departamento:")
print(df_inv_clean.groupby('Departamento')['Stock_Minimo'].value_counts())

## 3. Auditoría Post-Limpieza (Before vs. After)
Validamos estadísticamente la calidad final de los datasets limpios (`df.head()`, media de variables, nulos en 0).

In [ ]:
print("=== RESUMEN DE CALIDAD POST-LIMPIEZA ===")
print(f"Ventas:       {df_ventas_clean.shape[0]} registros | Nulos: {df_ventas_clean.isnull().sum().sum()}")
print(f"Detalle:      {df_detalle_clean.shape[0]} registros | Nulos: {df_detalle_clean.isnull().sum().sum()}")
print(f"Inventario:   {df_inv_clean.shape[0]} registros | Nulos: {df_inv_clean.isnull().sum().sum()}")

print("\nEstadísticas descriptivas limpias de Inventario (medias correctas):")
print(df_inv_clean[['Stock_Minimo', 'Stock_Actual', 'Costo_Unitario', 'Precio_Venta']].describe().round(2))

## 4. Exportación del Insumo Saneado a `datasets/limpio/`
Guardamos los archivos sin BOM ni columnas basura para alimentar al Cuaderno 1B (EDA y Clustering) y demás paneles del equipo.

In [ ]:
limpio_dir = 'datasets/limpio'
os.makedirs(limpio_dir, exist_ok=True)

df_ventas_clean.to_csv(os.path.join(limpio_dir, 'ventas.csv'), index=False, encoding='utf-8')
df_detalle_clean.to_csv(os.path.join(limpio_dir, 'detalle_ventas.csv'), index=False, encoding='utf-8')
df_detalle_clean.to_csv(os.path.join(limpio_dir, 'detalle-ventas.csv'), index=False, encoding='utf-8')
df_inv_clean.to_csv(os.path.join(limpio_dir, 'inventario.csv'), index=False, encoding='utf-8')

print("[OK] Datasets limpios y auditados exportados exitosamente a datasets/limpio/")